In [3]:
import os
import pandas as pd
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "zxzdxc86"))

folder = "E:/travel_data/citydata"

def write_batch(tx, rows, city):
    tx.run("""
        UNWIND $rows AS row

        MERGE (a:Attraction {name: row.name})
        SET a.url = row.url,
            a.address = row.address,
            a.intro = row.intro,
            a.open_time = row.open_time,
            a.image = row.image,
            a.rating = row.rating,
            a.duration = row.duration,
            a.season = row.season,
            a.ticket = row.ticket,
            a.tips = row.tips

        MERGE (c:City {name: $city})
        MERGE (a)-[:LOCATED_IN]->(c)
    """, rows=rows, city=city)


with driver.session() as session:
    for file in os.listdir(folder):
        if file.endswith(".csv"):

            city = file.replace(".csv", "")

            df = pd.read_csv(
                os.path.join(folder, file),
                encoding='utf-8-sig'
            )

            # 👇 核心：字段映射
            df = df.rename(columns={
                '名字': 'name',
                '链接': 'url',
                '地址': 'address',
                '介绍': 'intro',
                '开放时间': 'open_time',
                '图片链接': 'image',
                '评分': 'rating',
                '建议游玩时间': 'duration',
                '建议季节': 'season',
                '门票': 'ticket',
                '小贴士': 'tips'
            })

            rows = df.to_dict('records')
            session.execute_write(write_batch, rows, city)